In [1]:

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import rasterstats as rstats
import xarray as xr
import matplotlib.pyplot as plt

### Compare original and downscaled NEE, for all time

In [13]:
# Function to compute mean NEE for each MSA
def calculate_msa_mean(raster_file, msa_gdf):
    mean_values = []
    for _, row in msa_gdf.iterrows():
        msa_geometry = [row['geometry']]
        with rasterio.open(raster_file) as src:
            out_image, _ = mask(src, msa_geometry, crop=True, nodata=np.nan)
            out_image = out_image[0]
        
        # Compute mean ignoring NaNs
        mean_value = np.nanmean(out_image) if np.any(~np.isnan(out_image)) else np.nan
        mean_values.append(mean_value)
    return mean_values

def calc_MSA_NEE_mean_compare(year_month, msa_gdf_crsdownscaled, msa_gdf_crs4326, nee_orig_transform):
    '''
    Param:
        year_month(str): e.g.'200101'
    Return:
        two lists: 'nee_downscaled_msa_mean_{year_month}', 'nee_orig_msa_mean_{year_month}'
    '''
    print(f'Processing {year_month}...')

    # Read the downscaled NEE raster
    downscaled_nee_file_path = f'../gis/output/downscaledNEE/downscaledNEE_US_{year_month}.tif'
    with rasterio.open(downscaled_nee_file_path) as src:
        downscaled_nee = src.read(1)
        downscaled_nee = np.where(downscaled_nee == src.nodata, np.nan, downscaled_nee)

    # Compute mean NEE for downscaled data
    nee_downscaled_msa_mean = calculate_msa_mean(downscaled_nee_file_path, msa_gdf_crsdownscaled)


    # get original NEE value for specific year month
    original_nee_file_path = f'../gis/NEE/NEE.RS.FP-NONE.MLM-ALL.METEO-NONE.4320_2160.monthly.{year_month[:4]}.nc'
    variable = 'NEE'
    ds = xr.open_dataset(original_nee_file_path)
    original_nee = ds[variable][int(year_month[4:]) - 1].values  # Extract data for the specified month

    nee_orig_msa_mean = []

    # calculate mean original NEE for each MSA
    for index, row in msa_gdf_crs4326.iterrows():
        shape_info = row['geometry']
        name = row['NAMELSAD']
        # go through all geometries and compute zonal statistics
        res = rstats.zonal_stats(shape_info, original_nee, affine=nee_orig_transform, stats="mean")
        mean = res[0]['mean']
        # print(name, mean)
        nee_orig_msa_mean.append(mean)

    msa_mean_dict = {
        f'nee_downscaled_msa_mean_{year_month}': nee_downscaled_msa_mean,
        f'nee_orig_msa_mean_{year_month}': nee_orig_msa_mean
    }

    return msa_mean_dict

In [ ]:
# ====== prepare files and parameters like crs =====

msa_file = '../gis/msa/msaUS_mland_aea1_M1_all.shp' # TODO: change the file path to your MSA shapefile
# msa_gdf = gpd.read_file(msa_file)
msa_gdf_crsaea = gpd.read_file(msa_file)

# get NEE crs which is equal to GPP crs (EPSG:4326)
gpp_file = f'../gis/GPP_monthly_mean/gpp_200101.tif'
with rasterio.open(gpp_file) as gpp_dstrd:
    gpp_crs = gpp_dstrd.crs
nee_crs = gpp_crs


# get original NEE transform
minx, miny, maxx, maxy = -180.0, -90.0, 180.0, 90.0
nee_orig_resolution_x = 1/12
nee_orig_resolution_y = 1/12
nee_orig_transform = rasterio.transform.from_origin(minx, maxy, nee_orig_resolution_x, nee_orig_resolution_y)

with rasterio.open('../gis/output/downscaledNEE/downscaledNEE_US_200101.tif') as t:
    downscaled_nee_meta = t.meta.copy()
    
# Ensure CRS match between raster and shapefile
msa_gdf_crsdownscaled = msa_gdf_crsaea.to_crs(downscaled_nee_meta['crs'])

# make sure msa and nee are in same crs
msa_gdf_crs4326 = msa_gdf_crsaea.to_crs(nee_crs)

# ====== calculate =====
# add the mean values to the msa dataframe
msa_mean_compare = msa_gdf_crsaea[['NAMELSAD']]
for year in range(2001, 2016):
    for month in range(1, 13):
        year_month = f'{year}{month:02}'
        msa_mean_dict = calc_MSA_NEE_mean_compare(year_month, msa_gdf_crsdownscaled, msa_gdf_crs4326, nee_orig_transform)
        for key, value in msa_mean_dict.items():
            msa_mean_compare[key] = value

msa_mean_compare_file = '../gis/output/statistics/msa_mean_compare.csv'
msa_mean_compare.to_csv('../gis/output/statistics/msa_mean_compare.csv', index=False)

Processing 200101...


C:\Users\qifanw\AppData\Local\Temp\1\ipykernel_14800\2003268973.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  msa_mean_compare[key] = value
C:\Users\qifanw\AppData\Local\Temp\1\ipykernel_14800\2003268973.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  msa_mean_compare[key] = value


Processing 200102...


C:\Users\qifanw\AppData\Local\Temp\1\ipykernel_14800\2003268973.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  msa_mean_compare[key] = value
C:\Users\qifanw\AppData\Local\Temp\1\ipykernel_14800\2003268973.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  msa_mean_compare[key] = value


Processing 200103...


...

C:\Users\qifanw\AppData\Local\Temp\1\ipykernel_14800\2003268973.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  msa_mean_compare[key] = value
C:\Users\qifanw\AppData\Local\Temp\1\ipykernel_14800\2003268973.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  msa_mean_compare[key] = value


Processing 201511...


C:\Users\qifanw\AppData\Local\Temp\1\ipykernel_14800\2003268973.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  msa_mean_compare[key] = value
C:\Users\qifanw\AppData\Local\Temp\1\ipykernel_14800\2003268973.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  msa_mean_compare[key] = value


Processing 201512...


C:\Users\qifanw\AppData\Local\Temp\1\ipykernel_14800\2003268973.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  msa_mean_compare[key] = value
C:\Users\qifanw\AppData\Local\Temp\1\ipykernel_14800\2003268973.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  msa_mean_compare[key] = value


In [22]:
import pandas as pd

# Load the CSV file
df = pd.read_csv(msa_mean_compare_file)

# Extract the relevant columns for downscaled and original values
downscaled_cols = [col for col in df.columns if 'nee_downscaled_msa_mean' in col]
orig_cols = [col for col in df.columns if 'nee_orig_msa_mean' in col]

# Compute yearly means
yearly_mean_df = pd.DataFrame({'NAMELSAD': df['NAMELSAD']})

for year in range(2001, 2016):
    yearly_downscaled_cols = [col for col in downscaled_cols if str(year) in col]
    yearly_orig_cols = [col for col in orig_cols if str(year) in col]

    yearly_mean_df[f'nee_downscaled_yearly_mean_{year}'] = df[yearly_downscaled_cols].mean(axis=1)
    yearly_mean_df[f'nee_orig_yearly_mean_{year}'] = df[yearly_orig_cols].mean(axis=1)
    yearly_mean_df[f'nee_ratio_yearly_{year}'] = yearly_mean_df[f'nee_downscaled_yearly_mean_{year}'] / yearly_mean_df[f'nee_orig_yearly_mean_{year}']

# Compute mean for each month across all years
monthly_mean_df = pd.DataFrame({'NAMELSAD': df['NAMELSAD']})

for month in range(1, 13):
    month_str = f'{month:02d}'
    monthly_downscaled_cols = [col for col in downscaled_cols if col.endswith(month_str)]
    monthly_orig_cols = [col for col in orig_cols if col.endswith(month_str)]

    monthly_mean_df[f'nee_downscaled_monthly_mean_{month_str}'] = df[monthly_downscaled_cols].mean(axis=1)
    monthly_mean_df[f'nee_orig_monthly_mean_{month_str}'] = df[monthly_orig_cols].mean(axis=1)
    monthly_mean_df[f'nee_ratio_monthly_{month_str}'] = monthly_mean_df[f'nee_downscaled_monthly_mean_{month_str}'] / monthly_mean_df[f'nee_orig_monthly_mean_{month_str}']

In [31]:
1 / 1.2

0.8333333333333334

In [32]:
# Define the tolerance level
tolerance = 1.2

# Count occurrences where the yearly ratio deviates from 1
yearly_ratio_cols = [col for col in yearly_mean_df.columns if 'nee_ratio_yearly_' in col]
yearly_mean_df[f'exceeding_tolerance_{tolerance}_count'] = yearly_mean_df[yearly_ratio_cols].apply(lambda x: ((x < (1 / tolerance)) | (x > (tolerance))).sum(), axis=1)

# Count occurrences where the monthly ratio deviates from 1
monthly_ratio_cols = [col for col in monthly_mean_df.columns if 'nee_ratio_monthly_' in col]
monthly_mean_df[f'exceeding_tolerance_{tolerance}_count'] = monthly_mean_df[monthly_ratio_cols].apply(lambda x: ((x < (1 / tolerance)) | (x > (tolerance))).sum(), axis=1)


In [33]:
yearly_mean_df.to_csv('../gis/output/analysis/nee_downscale_orig_compare_yearly.csv', index=False)
monthly_mean_df.to_csv('../gis/output/analysis/nee_downscale_orig_compare_monthly.csv', index=False)

In [37]:
yearly_mean_df.sort_values(by=f'exceeding_tolerance_{tolerance}_count', ascending=False).head(20)[['NAMELSAD', f'exceeding_tolerance_{tolerance}_count']]


,NAMELSAD,exceeding_tolerance_1.2_count
325,"San Jose-Sunnyvale-Santa Clara, CA Metro Area",15
294,"Portland-Vancouver-Hillsboro, OR-WA Metro Area",15
259,"Monroe, LA Metro Area",15
73,"Yakima, WA Metro Area",15
338,"Spokane-Spokane Valley, WA Metro Area",15
309,"Yuba City, CA Metro Area",14
299,"Pueblo, CO Metro Area",9
157,"Davenport-Moline-Rock Island, IA-IL Metro Area",7
203,"Muncie, IN Metro Area",6
170,"Las Cruces, NM Metro Area",6


In [39]:
monthly_mean_df.sort_values(by=f'exceeding_tolerance_{tolerance}_count', ascending=False).head(20)[['NAMELSAD', f'exceeding_tolerance_{tolerance}_count']]


,NAMELSAD,exceeding_tolerance_1.2_count
325,"San Jose-Sunnyvale-Santa Clara, CA Metro Area",12
294,"Portland-Vancouver-Hillsboro, OR-WA Metro Area",12
309,"Yuba City, CA Metro Area",12
259,"Monroe, LA Metro Area",12
172,"Evansville, IN Metro Area",12
338,"Spokane-Spokane Valley, WA Metro Area",12
73,"Yakima, WA Metro Area",10
10,"Lake Havasu City-Kingman, AZ Metro Area",9
299,"Pueblo, CO Metro Area",6
16,"San Luis Obispo-Paso Robles, CA Metro Area",5


### save original nee from netcdf to tiff for comparison

In [44]:
year_month = '200601'
orig_nee_file = f'../gis/NEE/NEE.RS.FP-NONE.MLM-ALL.METEO-NONE.4320_2160.monthly.{year_month[:4]}.nc'

orig_nee_ds = xr.open_dataset(orig_nee_file)
orig_nee_time_file = f'../gis/output/originalNEE_{year_month}.tif'

time_slice = orig_nee_ds['NEE'][int(year_month[4:]) - 1].values

with rasterio.open(
    orig_nee_time_file,
    'w',
    driver='GTiff',
    height=time_slice.shape[0],
    width=time_slice.shape[1],
    count=1,
    dtype=str(time_slice.dtype),
    crs='EPSG:4326',  # Assuming WGS84 projection
    transform=nee_orig_transform,
) as dst:
    dst.write(time_slice, 1)